# Predict early-season FPL performance from the previous season

Build one row per player from 2023/24 performance, target mean `total_points` over the first 10 gameweeks of 2024/25, compare the four MVP approaches, then apply the selected model to 2024/25 aggregates to produce leakage-free scores for the 2025/26 ensemble.

> A player-level holdout is used because the target is now one value per player rather than one value per player-gameweek.


In [1]:
import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from fantasy_football.fpl_api.get_live_data import LivePlayerData
from fantasy_football.fpl_api.get_performance_data import get_most_recent_gw_points
from fantasy_football.model.base_models.pre_season import (
    DEFAULT_ARTIFACT_PATH,
    DEFAULT_CATEGORICAL_COLUMNS,
    DEFAULT_NUMERIC_FEATURES,
    DEFAULT_PRESEASON_SCORES_PATH,
    build_season_features,
)
from fantasy_football.model.training.utils import load_gw_data, load_player_data
from fantasy_football.utils import create_optimal_team

RANDOM_STATE = 42
EARLY_GAMEWEEKS = 10



## Load feature and target seasons


In [2]:
season_2023_24 = load_gw_data("2023-24")
season_2024_25 = load_gw_data("2024-25")

print("2023/24:", season_2023_24.shape)
print("2024/25:", season_2024_25.shape)


2023/24: (29725, 41)
2024/25: (27605, 49)


In [3]:
season_2023_24.columns

Index(['name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps',
       'clean_sheets', 'creativity', 'element', 'expected_assists',
       'expected_goal_involvements', 'expected_goals',
       'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored',
       'ict_index', 'influence', 'kickoff_time', 'minutes', 'opponent_team',
       'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards',
       'round', 'saves', 'selected', 'starts', 'team_a_score', 'team_h_score',
       'threat', 'total_points', 'transfers_balance', 'transfers_in',
       'transfers_out', 'value', 'was_home', 'yellow_cards', 'GW'],
      dtype='object')

# Feature list

In [4]:
mvp_numeric_features = list(DEFAULT_NUMERIC_FEATURES)

## Feature engineering

For each MVP numeric metric, calculate its season sum, appearance mean, appearance standard deviation, and exponentially decayed appearance mean. Later gameweeks receive more weight.


In [5]:
features_2023_24 = build_season_features(
    season_2023_24, feature_columns=mvp_numeric_features
)
features_2023_24.head()


,name,element,position,team,games_available,appearances,minutes_sum,assists_sum,assists_mean,assists_std,...,value_mean,value_std,value_decayed_mean,yellow_cards_sum,yellow_cards_mean,yellow_cards_std,yellow_cards_decayed_mean,minutes_mean,minutes_std,minutes_decayed_mean
0,Aaron Connolly,127,FWD,Brighton,37,0,0,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
1,Aaron Cresswell,530,DEF,West Ham,38,11,431,0,0.0,0.0,...,42.000000,0.000000,42.000000,1,0.090909,0.287480,0.062733,39.181818,33.275211,39.388661
2,Aaron Hickey,104,DEF,Brentford,37,9,713,0,0.0,0.0,...,45.000000,0.000000,45.000000,5,0.555556,0.496904,0.534602,79.222222,7.480015,79.624523
3,Aaron Ramsdale,17,GK,Arsenal,37,6,540,0,0.0,0.0,...,48.666667,1.972027,48.356664,0,0.000000,0.000000,0.000000,90.000000,0.000000,90.000000
4,Aaron Ramsey,675,MID,Burnley,36,13,522,0,0.0,0.0,...,50.000000,0.000000,50.000000,1,0.071429,0.257539,0.129663,37.285714,24.731208,40.884159


## Target: mean points in the first 10 gameweeks of 2024/25


In [6]:
early_2024_25 = season_2024_25.loc[
    season_2024_25["GW"].between(1, EARLY_GAMEWEEKS)
].copy()
player_gw_points = (
    early_2024_25.groupby(["name", "GW"], as_index=False)["total_points"].sum()
)
target_2024_25 = (
    player_gw_points.groupby("name", as_index=False)
    .agg(
        target_avg_points_early_gws=("total_points", "mean"),
        target_total_points_early_gws=("total_points", "sum"),
        target_gw_rows=("GW", "count"),
    )
)

model_data = features_2023_24.merge(target_2024_25, on="name", how="inner")
print(f"Matched {len(model_data):,} players across seasons")
model_data[["name", "target_avg_points_early_gws"]].sort_values(
    "target_avg_points_early_gws", ascending=False
).head(10)


Matched 472 players across seasons


,name,target_avg_points_early_gws
336,Mohamed Salah,9.3
84,Cole Palmer,8.2
134,Erling Haaland,7.7
61,Bryan Mbeumo,7.1
79,Chris Wood,6.6
62,Bukayo Saka,6.5
99,Danny Welbeck,6.2
289,Luis Díaz,6.1
357,Nicolas Jackson,5.8
369,Ollie Watkins,5.3


## Player-level train/validation split


In [7]:
target_column = "target_avg_points_early_gws"
categorical_columns = list(DEFAULT_CATEGORICAL_COLUMNS)
excluded_columns = {
    "name", "element", target_column, "target_total_points_early_gws", "target_gw_rows"
}
model_features = [column for column in model_data.columns if column not in excluded_columns]

train_index, valid_index = train_test_split(
    model_data.index, test_size=0.20, random_state=RANDOM_STATE
)
X_train = model_data.loc[train_index, model_features].copy()
X_valid = model_data.loc[valid_index, model_features].copy()
y_train = model_data.loc[train_index, target_column].copy()
y_valid = model_data.loc[valid_index, target_column].copy()
for frame in (X_train, X_valid):
    frame[categorical_columns] = frame[categorical_columns].fillna("__MISSING__").astype(str)

print(f"Train players: {len(X_train):,}; validation players: {len(X_valid):,}")


Train players: 377; validation players: 95


## Four modelling approaches from the MVP


In [8]:
models = {}
validation_predictions = {}

dummy_model = DummyRegressor(strategy="mean").fit(X_train, y_train)
models["Dummy"] = dummy_model
validation_predictions["Dummy"] = dummy_model.predict(X_valid)

def fit_catboost(X, y, sample_weight=None):
    model = CatBoostRegressor(
        iterations=163, learning_rate=0.03, depth=6, loss_function="RMSE",
        random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False,
    )
    model.fit(X, y, cat_features=categorical_columns, sample_weight=sample_weight)
    return model

unweighted_model = fit_catboost(X_train, y_train)
models["Unweighted"] = unweighted_model
validation_predictions["Unweighted"] = unweighted_model.predict(X_valid)

basic_weights = pd.Series(1.0, index=y_train.index)
basic_weights.loc[y_train > 6] = 4.0
basic_model = fit_catboost(X_train, y_train, basic_weights)
models["BASIC weighting"] = basic_model
validation_predictions["BASIC weighting"] = basic_model.predict(X_valid)

target_percentiles = y_train.rank(pct=True, method="average")
percentile_weights = pd.cut(
    target_percentiles, bins=[0, .25, .50, .75, .90, 1.0],
    labels=[1.0, 1.5, 2.0, 3.0, 4.0], include_lowest=True,
).astype(float)
percentile_model = fit_catboost(X_train, y_train, percentile_weights)
models["Percentile weighting"] = percentile_model
validation_predictions["Percentile weighting"] = percentile_model.predict(X_valid)


## Comparison table


In [9]:
def evaluate_predictions(predictions, top_fraction=0.10):
    evaluation = pd.DataFrame({"actual": y_valid, "predicted": predictions})
    n = max(1, int(np.ceil(len(evaluation) * top_fraction)))
    predicted_top = evaluation.nlargest(n, "predicted")
    actual_top_indices = set(evaluation.nlargest(n, "actual").index)
    return {
        "MAE": mean_absolute_error(evaluation["actual"], evaluation["predicted"]),
        "RMSE": root_mean_squared_error(evaluation["actual"], evaluation["predicted"]),
        "R2": r2_score(evaluation["actual"], evaluation["predicted"]),
        "top_decile_avg_actual_points": predicted_top["actual"].mean(),
        "top_decile_hit_rate": predicted_top.index.isin(actual_top_indices).mean(),
        "top_decile_oracle_regret": (
            evaluation.nlargest(n, "actual")["actual"].mean()
            - predicted_top["actual"].mean()
        ),
    }

comparison_table = (
    pd.DataFrame.from_dict(
        {name: evaluate_predictions(preds) for name, preds in validation_predictions.items()},
        orient="index",
    )
    .rename_axis("model").reset_index()
    .sort_values(["top_decile_avg_actual_points", "MAE"], ascending=[False, True])
    .reset_index(drop=True)
)
selected_model_name = comparison_table.loc[0, "model"]
comparison_table.round(3)


,model,MAE,RMSE,R2,top_decile_avg_actual_points,top_decile_hit_rate,top_decile_oracle_regret
0,BASIC weighting,0.874,1.289,0.222,2.34,0.3,1.96
1,Percentile weighting,0.970,1.305,0.202,2.34,0.3,1.96
2,Unweighted,0.855,1.241,0.278,2.14,0.2,2.16
3,Dummy,1.257,1.472,-0.015,1.71,0.1,2.59


In [10]:
print(f"Best validation model is {selected_model_name}")

Best validation model is BASIC weighting


In [11]:
# Refit the chosen percentile-weighted approach on all development data.
X_full = model_data[model_features].copy()
y_full = model_data[target_column].copy()
X_full[categorical_columns] = (
    X_full[categorical_columns].fillna("__MISSING__").astype(str)
)

full_target_percentiles = y_full.rank(pct=True, method="average")
full_percentile_weights = pd.cut(
    full_target_percentiles,
    bins=[0, .25, .50, .75, .90, 1.0],
    labels=[1.0, 1.5, 2.0, 3.0, 4.0],
    include_lowest=True,
).astype(float)
final_preseason_model = fit_catboost(
    X_full, y_full, sample_weight=full_percentile_weights
)

DEFAULT_ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "model": final_preseason_model,
        "model_name": "Percentile weighting",
        "feature_columns": model_features,
        "categorical_columns": categorical_columns,
    },
    DEFAULT_ARTIFACT_PATH,
)
print(f"Saved pre-season model to {DEFAULT_ARTIFACT_PATH}")

# Produce player scores for the 2025/26 ensemble from 2024/25 features.
features_2024_25 = build_season_features(
    season_2024_25, feature_columns=mvp_numeric_features
)
X_2025_26 = features_2024_25.reindex(columns=model_features).copy()
X_2025_26[categorical_columns] = (
    X_2025_26[categorical_columns].fillna("__MISSING__").astype(str)
)
players_2024_25 = load_player_data("2024-25")[["id", "code"]].copy()
preseason_scores_2025_26 = features_2024_25[["element", "name"]].merge(
    players_2024_25, left_on="element", right_on="id", how="left"
)
assert preseason_scores_2025_26["code"].notna().all()
preseason_scores_2025_26["preseason_model_score"] = (
    final_preseason_model.predict(X_2025_26)
)
joblib.dump(preseason_scores_2025_26, DEFAULT_PRESEASON_SCORES_PATH)
print(f"Saved 2025/26 pre-season scores to {DEFAULT_PRESEASON_SCORES_PATH}")


Saved pre-season model to /Users/calumthompson/Documents/fantasy_football_v2/fantasy_football/model/artifacts/pre_season_model.joblib


Saved 2025/26 pre-season scores to /Users/calumthompson/Documents/fantasy_football_v2/fantasy_football/model/artifacts/pre_season_scores_2025_26.joblib
